In [1]:
import gzip
import dpkt
import struct
from collections import defaultdict
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
import glob
from pathlib import Path
import time
import json
import os
import re

In [10]:
gz_path = r"C:\IEX\cache\data_feeds*.pcap.gz"

volume_by_symbol = defaultdict(int)
trades_by_symbol = defaultdict(int)

file_path = glob.glob(gz_path)[0]

In [3]:
big_names = ['NU','GRAB', 'NVDA', 'AGNC', 'INTC', 'RIG', 'NFLX','NOK', 'CLF', 'KO']
print(len(big_names))

10


In [4]:
big_names = set(big_names)

output_dir = r"C:\IEX\20261009"
os.makedirs(output_dir, exist_ok=True)

writers = {}
buffers = {}

columns = [
    "timestamp",
    "symbol",
    "price",
    "size",
    "trade_id",
    "flags",
    "packet_no"
]

In [5]:
def get_writer(symbol):

    if symbol not in writers:
        schema = pa.schema([
            ("timestamp", pa.int64()),
            ("symbol", pa.string()),
            ("price", pa.float64()),
            ("size", pa.int64()),
            ("trade_id", pa.int64()),
            ("flags", pa.int64()),
            ("packet_no", pa.int64()),
        ])

        path = os.path.join(output_dir, f"{symbol}_trades.parquet")
        writers[symbol] = pq.ParquetWriter(path, schema, compression="snappy")
        buffers[symbol] = []

    return writers[symbol]

In [7]:
OUTPUT_DIR = Path("iex_trade_chunks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"
CHECKPOINT_EVERY = 1_000_000


def save_checkpoint(last_packet, chunk_no, total_trades):
    """Atomically save extraction progress."""

    state = {
        "last_packet": last_packet,
        "next_chunk": chunk_no,
        "total_trades": total_trades,
        "input_file": str(Path(file_path).resolve()),
        "input_size": os.path.getsize(file_path),
        "input_mtime_ns": os.stat(file_path).st_mtime_ns,
    }

    temp_file = CHECKPOINT_FILE.with_suffix(".tmp")

    with open(temp_file, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp_file, CHECKPOINT_FILE)


def load_checkpoint():
    """Restore state, or initialize a fresh run."""

    if not CHECKPOINT_FILE.exists():
        return -1, {s: 0 for s in big_names}, {
            s: 0 for s in big_names
        }

    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        state = json.load(f)

    # Prevent resuming against a different input file.
    if (
        state["input_file"] != str(Path(file_path).resolve())
        or state["input_size"] != os.path.getsize(file_path)
        or state["input_mtime_ns"] != os.stat(file_path).st_mtime_ns
    ):
        raise ValueError(
            "Input PCAP has changed. Use a separate output directory "
            "or explicitly start a new extraction."
        )

    last_packet = state["last_packet"]

    chunk_no = {
        s: state["next_chunk"].get(s, 0)
        for s in big_names
    }

    total_trades = {
        s: state["total_trades"].get(s, 0)
        for s in big_names
    }

    return last_packet, chunk_no, total_trades

In [8]:
last_packet, chunk_no, total_trades = load_checkpoint()

# Remove incomplete temporary files.
for path in OUTPUT_DIR.glob("*.tmp"):
    path.unlink()

# Remove chunks not committed by the checkpoint.
for symbol in big_names:
    pattern = re.compile(
        rf"^{re.escape(symbol)}_part_(\d+)\.parquet$"
    )

    for path in OUTPUT_DIR.glob(f"{symbol}_part_*.parquet"):
        match = pattern.match(path.name)

        if match and int(match.group(1)) >= chunk_no[symbol]:
            path.unlink()

buffers = {s: [] for s in big_names}

print("Resume after packet:", f"{last_packet:,}")
print("Next chunk numbers:", chunk_no)
print("Previously saved trades:", total_trades)

Resume after packet: -1
Next chunk numbers: {'INTC': 0, 'NOK': 0, 'CLF': 0, 'AGNC': 0, 'KO': 0, 'GRAB': 0, 'NFLX': 0, 'RIG': 0, 'NVDA': 0, 'NU': 0}
Previously saved trades: {'INTC': 0, 'NOK': 0, 'CLF': 0, 'AGNC': 0, 'KO': 0, 'GRAB': 0, 'NFLX': 0, 'RIG': 0, 'NVDA': 0, 'NU': 0}


In [ ]:
BUFFER_SIZE = 10_000
PROGRESS_EVERY = 1_000_000

OUTPUT_DIR = Path("iex_trade_chunks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

buffers = {symbol: [] for symbol in big_names}
chunk_no = {symbol: 0 for symbol in big_names}

total_trades = {symbol: 0 for symbol in big_names}
last_progress = time.time()

def flush_buffer(symbol):
    buffer = buffers[symbol]

    if not buffer:
        return
    chunk = chunk_no[symbol]
    filename = OUTPUT_DIR / (f"{symbol}_part_{chunk:06d}.parquet")
    temp_filename = filename.with_suffix(".tmp")
    table = pa.Table.from_pylist(buffer)
    pq.write_table(
        table,
        temp_filename,
        compression="snappy"
    )
    os.replace(temp_filename, filename)
    total_trades[symbol] += len(buffer)
    chunk_no[symbol] += 1

    print(f"Saved {filename.name}: {len(buffer):,} trades")
    buffer.clear()

packet_no = last_packet
last_packet_seen = last_packet

with gzip.open(file_path, "rb") as f:
    pcap = dpkt.pcapng.Reader(f)

    for packet_no, (pcap_ts, raw_packet) in enumerate(pcap):

        if packet_no <= last_packet:
            continue

        # Parse this packet and append trades to buffers.
        # Do not put the checkpoint logic inside this parsing block.
        try:
            eth = dpkt.ethernet.Ethernet(raw_packet)
            ip = eth.data

            if isinstance(ip, dpkt.ip.IP):
                udp = ip.data

                if isinstance(udp, dpkt.udp.UDP):
                    payload = udp.data

                    if len(payload) >= 40:
                        message_count = struct.unpack_from(
                            "<H", payload, 14
                        )[0]
                        data = payload[40:]
                        pos = 0

                        for _ in range(message_count):
                            if pos + 2 > len(data):
                                break

                            msg_len = struct.unpack_from("<H", data, pos)[0]
                            msg_start = pos + 2
                            msg_end = msg_start + msg_len

                            if msg_end > len(data):
                                break

                            msg = data[msg_start:msg_end]
                            pos = msg_end

                            if len(msg) < 38 or msg[0] != ord("T"):
                                continue

                            symbol = (
                                msg[10:18]
                                .decode("ascii", errors="ignore")
                                .rstrip("\x00")
                                .strip()
                            )

                            if symbol not in buffers:
                                continue

                            timestamp = struct.unpack_from("<Q", msg, 2)[0]
                            size = struct.unpack_from("<I", msg, 18)[0]
                            price = struct.unpack_from("<q", msg, 22)[0] / 10000.0
                            trade_id = struct.unpack_from("<Q", msg, 30)[0]

                            buffers[symbol].append({
                                "timestamp": timestamp,
                                "symbol": symbol,
                                "price": price,
                                "size": size,
                                "trade_id": trade_id,
                                "flags": msg[1],
                                "packet_no": packet_no,
                            })

                            if len(buffers[symbol]) >= BUFFER_SIZE:
                                flush_buffer(symbol)

        except (dpkt.UnpackError, struct.error, IndexError):
            print(f"Malformed packet skipped: {packet_no}")

        # This part now runs for every packet, even if it wasn't UDP.
        last_packet_seen = packet_no

        if (packet_no + 1) % CHECKPOINT_EVERY == 0:
            for symbol in buffers:
                flush_buffer(symbol)

            save_checkpoint(
                packet_no,
                chunk_no,
                total_trades
            )

            print(f"Checkpoint saved after packet {packet_no:,}")

# Final flush
for symbol in buffers:
    flush_buffer(symbol)

if last_packet_seen >= 0:
    save_checkpoint(last_packet_seen, chunk_no, total_trades)

print("Extraction completed.")

Saved INTC_part_000000.parquet: 111 trades
Saved NOK_part_000000.parquet: 38 trades
Saved CLF_part_000000.parquet: 3 trades
Saved AGNC_part_000000.parquet: 22 trades
Saved KO_part_000000.parquet: 37 trades
Saved GRAB_part_000000.parquet: 49 trades
Saved NFLX_part_000000.parquet: 87 trades
Saved RIG_part_000000.parquet: 2 trades
Saved NVDA_part_000000.parquet: 218 trades
Saved NU_part_000000.parquet: 43 trades
Checkpoint saved after packet 999,999
Saved INTC_part_000001.parquet: 141 trades
Saved NOK_part_000001.parquet: 15 trades
Saved CLF_part_000001.parquet: 17 trades
Saved AGNC_part_000001.parquet: 13 trades
Saved KO_part_000001.parquet: 36 trades
Saved GRAB_part_000001.parquet: 55 trades
Saved NFLX_part_000001.parquet: 40 trades
Saved RIG_part_000001.parquet: 2 trades
Saved NVDA_part_000001.parquet: 127 trades
Saved NU_part_000001.parquet: 58 trades
Checkpoint saved after packet 1,999,999
Saved INTC_part_000002.parquet: 139 trades
Saved NOK_part_000002.parquet: 4 trades
Saved CLF_pa

In [ ]:
1+1

In [12]:
import pyarrow.dataset as ds

nvda = ds.dataset(
    "iex_trade_chunks",
    format="parquet"
).to_table(
    filter=ds.field("symbol") == "NVDA"
).to_pandas()

nvda = nvda.sort_values("timestamp").reset_index(drop=True)

ArrowInvalid: Could not open Parquet input source 'iex_trade_chunks/checkpoint.json': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.